## 一、 核心组件加载：AutoModelForCausalLM

## 1.1 为什么是 CausalLM？

大语言模型（LLM）大多属于 **Causal Language Model（因果语言模型）**。

- **因果性**：模型在预测第 $n$ 个词时，只能看到前 $n-1$ 个词，不能“偷看”后面的信息。
    
- **核心任务**：根据上文预测下一个词的概率分布。
    
- **类比**：就像玩接龙游戏，模型本质上是一个超级复杂的“概率预测器”。
    

## 1.2 加载模型权重

在 `transformers` 库中，`AutoModelForCausalLM` 是最通用的加载器。

In [20]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# 设置模型路径（使用上一章下载的本地路径）
model_path = "C:\\0store\\LLM\\Qwen2.5-1.5B" 

# 加载分词器
tokenizer = AutoTokenizer.from_pretrained(model_path)

# 加载模型
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    device_map="auto",          # 🚀 关键参数：自动检测显卡并分配权重
    torch_dtype=torch.bfloat16, # 🚀 关键参数：以 BF16 精度加载，显存占用减半
)
!nvidia-smi
print("✅ 模型加载成功！")


Mon Mar 23 18:23:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 560.94                 Driver Version: 560.94         CUDA Version: 12.6     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4070 ...  WDDM  |   00000000:01:00.0  On |                  N/A |
|  0%   47C    P2             31W /  220W |    7726MiB /  12282MiB |      2%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

#### 🛠️ 关键工程参数解析：

1. **`device_map="auto"`**:
    
    - **作用**：自动计算模型每一层的大小，并将其分配到可用的 GPU 上。如果单张显卡装不下，它会自动尝试将剩余部分放入 CPU 内存（尽管推理会变慢）。
        
2. **`torch_dtype=torch.bfloat16`**:
    
    - **原理**：默认权重通常是 FP32（4字节/参数）。1.5B 模型全量加载需要 $1.5 \times 4 = 6\text{GB}$ 显存。
        
    - **优化**：使用 `bfloat16`（2字节/参数），显存需求降至约 $3\text{GB}$，且几乎不损失精度。

## 二、 模型的“手术台”：解剖结构

### 2.1 打印模型对象

直接打印 `model`，我们可以看到 Qwen 这种标准 Transformer 架构的三个物理部分。

In [21]:
print(model)

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((1536,), eps=1e-06)
    (rotar

**你会观察到以下层级：**

- **`model.embed_tokens` (Embedding)**: 词嵌入层。负责将 Token ID 映射为高维向量（类似于给词语分配坐标）。
    
- **`model.layers` (Blocks 0-27)**: 28 个重复的 **Decoder Layer**。
    
    - 每个 Layer 包含 **Self-Attention**（负责关联上下文）和 **MLP/FFN**（负责存储知识和进行非线性变换）。
        
- **`lm_head`**: 最后的输出投影层。负责将处理后的向量映射回词表大小（如 151,936 个词），算出每个词出现的概率。
    

### 2.2 查看 Config 映射

模型的“说明书”存放在 `config.json` 中。

In [22]:
# 获取模型配置
config = model.config

print(f"模型层数 (num_hidden_layers): {config.num_hidden_layers}")
print(f"隐藏层维度 (hidden_size): {config.hidden_size}")
print(f"词表大小 (vocab_size): {config.vocab_size}")
print(f"最大上下文长度 (max_position_embeddings): {config.max_position_embeddings}")

模型层数 (num_hidden_layers): 28
隐藏层维度 (hidden_size): 1536
词表大小 (vocab_size): 151936
最大上下文长度 (max_position_embeddings): 131072


## 三、 第一次推理实战 (Forward Pass)

这一步我们不使用封装好的工具，而是像“手动挡”一样驱动模型。

### 3.1 准备输入

将文字转为张量，并手动搬运到 GPU。
这里我们需要理解，在使用tokenizer的时候是在用CPU处理，而大模型的加载是自动到了GPU上的。所以我们需要手动将输入搬到GPU上防止设备不一致无法计算的问题。

任何的张量使用 `.to("cuda")`可以指定设备。

In [23]:
text = "人工智能的本质是"
inputs = tokenizer(text, return_tensors="pt").to("cuda")

print(f"输入张量形状: {inputs['input_ids'].shape}") # [Batch_Size, Seq_Len]
print(f"输入张量的设备：", inputs['input_ids'].device)

输入张量形状: torch.Size([1, 3])
输入张量的设备： cuda:0


### 3.2 封装好的生成工具：generate()

在实际开发中，我们不会手动去算 Logits，而是使用 generate() 函数，它会自动循环：预测词 -> 拼接词 -> 再预测词。

In [26]:
# 简单的生成调用
output_ids = model.generate(
    inputs["input_ids"], 
    max_new_tokens=50,
    temperature=0.7,
)

print(tokenizer.decode(output_ids[0], skip_special_tokens=True))

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


人工智能的本质是____。
A. 机器学习
B. 机器思维
C. 机器智能
D. 机器意识
答案:
C

在进行高处作业时，除有关人员外，不准他人在工作地点的下面通行



| **参数名**              | **类型**  | **作用**                                            |
| -------------------- | ------- | ------------------------------------------------- |
| **`max_new_tokens`** | `int`   | **限制长度**。规定模型最多吐出多少个新词。                           |
| **`temperature`**    | `float` | **创造力开关**。值越高，输出越随机（更有趣）；值越低，输出越稳定（更刻板）。通常设为 0.7。 |
| **`top_p`**          | `float` | **核心采样**。只在累计概率达到 p 的最小词组中采样，兼顾多样性与合理性。           |
| **`do_sample`**      | `bool`  | **是否采样**。若为 `False`，则每次都选概率最高的词（贪婪搜索）。            |



## 🚀 动手尝试

- 修改 `temperature` 为 `2.0`（极大值）和 `0.1`（极小值），观察同一个问题下模型输出稳定性的变化。
    
- 尝试输入一段代码片段，看看 1.5B 的小模型是否具备基础的代码续写能力。

- 可以尝试切换别的模型，看看生成有什么不同。也可以看看同一模型的不同版本，如'Qwen2.5-1.5B-Instruct'和`Qwen2.5-1.5B`指令微调的区别。